### Count Manuscripts and Pages

# Exploratory Data Analysis On Competition Dataset

In [1]:
from pathlib import Path
import json
from collections import Counter

# Adjust this path to your dataset location
dataset_path = Path("/Users/annanya/Desktop/college/sem8/computer vision/ICDAR2026/cmmhwr26").expanduser()

In [2]:
# Get all manuscript folders
manuscript_folders = [d for d in dataset_path.iterdir() if d.is_dir()]

print(f"Total manuscript folders: {len(manuscript_folders)}")

Total manuscript folders: 300


In [3]:
# Count pages per manuscript
total_pages = 0
pages_per_manuscript = []

for manuscript in manuscript_folders:
    xml_files = list(manuscript.glob("*.xml"))
    pages_per_manuscript.append(len(xml_files))
    total_pages += len(xml_files)

print(f"Total pages (XML files): {total_pages}")
print(f"Average pages per manuscript: {total_pages / len(manuscript_folders):.1f}")
print(f"Min pages: {min(pages_per_manuscript)}")
print(f"Max pages: {max(pages_per_manuscript)}")

Total pages (XML files): 2236
Average pages per manuscript: 7.5
Min pages: 1
Max pages: 75


### Count Lines per Language

In [4]:
import xml.etree.ElementTree as ET
from collections import Counter

language_line_counts = Counter()
language_page_counts = Counter()

for manuscript_folder in dataset_path.iterdir():
    if not manuscript_folder.is_dir():
        continue
    
    # Read metadata
    metadata_file = manuscript_folder / "metadata.json"
    if not metadata_file.exists():
        print(f"Warning: No metadata in {manuscript_folder.name}")
        continue
    
    with open(metadata_file) as f:
        metadata = json.load(f)
        language = metadata.get("language", "Unknown")
    
    # Count lines in all XML files in this manuscript
    xml_files = list(manuscript_folder.glob("*.xml"))
    language_page_counts[language] += len(xml_files)
    
    for xml_file in xml_files:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        # Count TextLine elements (namespace aware)
        text_lines = root.findall('.//{*}TextLine')
        language_line_counts[language] += len(text_lines)

print("\n=== Lines per Language ===")
print(f"{'Language':<15} {'Pages':<10} {'Lines':<10} {'Lines/Page':<10}")
print("-" * 50)
total_lines = 0
total_pages = 0
for language in sorted(language_line_counts.keys()):
    lines = language_line_counts[language]
    pages = language_page_counts[language]
    total_lines += lines
    total_pages += pages
    print(f"{language:<15} {pages:<10} {lines:<10} {lines/pages:>9.1f}")

print("-" * 50)
print(f"{'TOTAL':<15} {total_pages:<10} {total_lines:<10} {total_lines/total_pages:>9.1f}")


=== Lines per Language ===
Language        Pages      Lines      Lines/Page
--------------------------------------------------
Castilian       448        33805           75.5
Catalan         38         1597            42.0
French          834        58002           69.5
Gallician       7          547             78.1
Italian         172        8112            47.2
Latin           720        48162           66.9
Navarrese       6          739            123.2
Venitian        11         261             23.7
--------------------------------------------------
TOTAL           2236       151225          67.6


### Analyze Line Types

In [5]:
line_type_counts = Counter()

# Sample 50 random XML files
import random
xml_files = list(dataset_path.rglob("*.xml"))
sample_files = random.sample(xml_files, min(50, len(xml_files)))

for xml_file in sample_files:
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # Find all TextLine elements
    for text_line in root.findall('.//{*}TextLine'):
        # Check TAGREFS attribute (indicates line type)
        tag_refs = text_line.get('TAGREFS', '')
        if 'LT1894' in tag_refs or 'DefaultLine' in tag_refs:
            line_type_counts['DefaultLine'] += 1
        elif 'LT1895' in tag_refs or 'InterlinearLine' in tag_refs:
            line_type_counts['InterlinearLine'] += 1
        else:
            line_type_counts['Other'] += 1

print("\n=== Line Type Distribution ===")
for line_type, count in line_type_counts.most_common():
    print(f"{line_type}: {count} ({count/sum(line_type_counts.values())*100:.1f}%)")


=== Line Type Distribution ===
Other: 4535 (100.0%)


### Verify Image-XML correspondence

In [8]:
mismatches = []

for manuscript_folder in dataset_path.iterdir():
    if not manuscript_folder.is_dir():
        continue
    
    xml_files = set(f.stem for f in manuscript_folder.glob("*.xml"))
    jpg_files = set(f.stem for f in manuscript_folder.glob("*.jpg"))
    jpeg_files = set(f.stem for f in manuscript_folder.glob("*.jpeg"))
    JPEG_files = set(f.stem for f in manuscript_folder.glob("*.JPEG"))
    png_files = set(f.stem for f in manuscript_folder.glob("*.png"))
    
    image_files = jpg_files | png_files | jpeg_files | JPEG_files
    
    if xml_files != image_files:
        mismatches.append({
            'folder': manuscript_folder.name,
            'xml_only': xml_files - image_files,
            'image_only': image_files - xml_files
        })

if mismatches:
    print(f"\nFound {len(mismatches)} folders with mismatches:")
    for m in mismatches[:10]:  # Show first 10
        print(f"  {m['folder']}: XML={len(m['xml_only'])}, IMG={len(m['image_only'])}")
else:
    print("\n✅ All XML files have corresponding images!")


✅ All XML files have corresponding images!


### Sample Line Extraction Test

In [10]:
from PIL import Image

# Get first XML file
first_xml = list(dataset_path.rglob("*.xml"))[0]
manuscript_folder = first_xml.parent
image_file = manuscript_folder / f"{first_xml.stem}.jpg"

if not image_file.exists():
    image_file = manuscript_folder / f"{first_xml.stem}.png"

print(f"Testing: {first_xml.name}")

# Parse XML
tree = ET.parse(first_xml)
root = tree.getroot()

# Load image
page_image = Image.open(image_file)
print(f"Page size: {page_image.size}")

# Extract first 5 lines
text_lines = root.findall('.//{*}TextLine')[:5]

for i, text_line in enumerate(text_lines):
    line_id = text_line.get('ID')
    hpos = float(text_line.get('HPOS'))
    vpos = float(text_line.get('VPOS'))
    width = float(text_line.get('WIDTH'))
    height = float(text_line.get('HEIGHT'))
    
    # Get text content
    string_elem = text_line.find('.//{*}String')
    text = string_elem.get('CONTENT') if string_elem is not None else ''
    
    print(f"\nLine {i+1} [{line_id}]:")
    print(f"  Position: ({hpos}, {vpos}), Size: {width}x{height}")
    print(f"  Text: {text}")
    
    # Crop line from page
    line_image = page_image.crop((hpos, vpos, hpos+width, vpos+height))
    
    # Save sample
    line_image.save(f"sample_line_{i+1}.jpg")
    print(f"  Saved: sample_line_{i+1}.jpg")

print("\n✅ Line extraction working! Check the saved images.")

Testing: page-008.xml
Page size: (3009, 4050)

Line 1 [eSc_line_3d9adcf9]:
  Position: (166.0, 270.0), Size: 572.0x83.0
  Text: ⁊ faite solonc sa maistrie
  Saved: sample_line_1.jpg

Line 2 [eSc_line_ed03ab12]:
  Position: (164.0, 346.0), Size: 540.0x68.0
  Text: de la maniere del ourier
  Saved: sample_line_2.jpg

Line 3 [eSc_line_4670ffaf]:
  Position: (164.0, 409.0), Size: 558.0x68.0
  Text: Q/ maniere a de recourier
  Saved: sample_line_3.jpg

Line 4 [eSc_line_8b7ed31e]:
  Position: (166.0, 470.0), Size: 515.0x65.0
  Text: tot alsi nature se done
  Saved: sample_line_4.jpg

Line 5 [eSc_line_dbcde090]:
  Position: (165.0, 526.0), Size: 557.0x68.0
  Text: La ou dex ueut ⁊ sabandone
  Saved: sample_line_5.jpg

✅ Line extraction working! Check the saved images.


### Character Set Analysis

In [11]:
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter
import unicodedata

all_chars = Counter()
special_chars = set()
language_chars = {}

for manuscript_folder in dataset_path.iterdir():
    if not manuscript_folder.is_dir():
        continue
    
    # Get language
    import json
    metadata_file = manuscript_folder / "metadata.json"
    if metadata_file.exists():
        with open(metadata_file) as f:
            language = json.load(f).get("language", "Unknown")
    else:
        language = "Unknown"
    
    if language not in language_chars:
        language_chars[language] = Counter()
    
    # Process all XML files
    for xml_file in manuscript_folder.glob("*.xml"):
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        for text_line in root.findall('.//{*}TextLine'):
            string_elem = text_line.find('.//{*}String')
            if string_elem is not None:
                text = string_elem.get('CONTENT', '')
                
                for char in text:
                    all_chars[char] += 1
                    language_chars[language][char] += 1
                    
                    # Track special/non-ASCII characters
                    if ord(char) > 127:  # Non-ASCII
                        special_chars.add(char)

print("\n=== Character Set Analysis ===")
print(f"Total unique characters: {len(all_chars)}")
print(f"Special (non-ASCII) characters: {len(special_chars)}")

print("\n=== Top 50 Most Common Characters ===")
for char, count in all_chars.most_common(50):
    char_name = unicodedata.name(char, 'UNKNOWN')
    print(f"'{char}' (U+{ord(char):04X}): {count:6d} - {char_name}")

print("\n=== Special Characters (Non-ASCII) ===")
special_sorted = sorted(special_chars, key=lambda c: all_chars[c], reverse=True)
for char in special_sorted[:50]:  # Top 50 special chars
    char_name = unicodedata.name(char, 'UNKNOWN')
    print(f"'{char}' (U+{ord(char):04X}): {all_chars[char]:6d} - {char_name}")

print("\n=== Character Set Size by Language ===")
for lang in sorted(language_chars.keys()):
    unique = len(language_chars[lang])
    total = sum(language_chars[lang].values())
    print(f"{lang:<15} Unique: {unique:4d}  Total: {total:8d}")

# Save full character inventory
with open("character_inventory.txt", "w", encoding="utf-8") as f:
    f.write("Complete Character Inventory\n")
    f.write("="*50 + "\n\n")
    for char, count in all_chars.most_common():
        char_name = unicodedata.name(char, 'UNKNOWN')
        f.write(f"'{char}' U+{ord(char):04X} {char_name}: {count}\n")

print("\n✅ Saved complete inventory to character_inventory.txt")


=== Character Set Analysis ===
Total unique characters: 229
Special (non-ASCII) characters: 154

=== Top 50 Most Common Characters ===
' ' (U+0020): 866474 - SPACE
'e' (U+0065): 575259 - LATIN SMALL LETTER E
'a' (U+0061): 378811 - LATIN SMALL LETTER A
'i' (U+0069): 378794 - LATIN SMALL LETTER I
's' (U+0073): 338038 - LATIN SMALL LETTER S
'o' (U+006F): 304200 - LATIN SMALL LETTER O
'u' (U+0075): 283037 - LATIN SMALL LETTER U
't' (U+0074): 282799 - LATIN SMALL LETTER T
'n' (U+006E): 270042 - LATIN SMALL LETTER N
'r' (U+0072): 254784 - LATIN SMALL LETTER R
'l' (U+006C): 211763 - LATIN SMALL LETTER L
'd' (U+0064): 158311 - LATIN SMALL LETTER D
'c' (U+0063): 155734 - LATIN SMALL LETTER C
'm' (U+006D): 124900 - LATIN SMALL LETTER M
'̃' (U+0303): 117633 - COMBINING TILDE
'p' (U+0070): 101558 - LATIN SMALL LETTER P
'.' (U+002E):  73259 - FULL STOP
'q' (U+0071):  59605 - LATIN SMALL LETTER Q
'g' (U+0067):  51399 - LATIN SMALL LETTER G
'b' (U+0062):  50731 - LATIN SMALL LETTER B
'f' (U+0066):  

### Text Length Distribution

In [12]:
# text_length_analysis.py
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import defaultdict
import json

line_lengths = []
line_lengths_by_language = defaultdict(list)
word_counts = []

for manuscript_folder in dataset_path.iterdir():
    if not manuscript_folder.is_dir():
        continue
    
    # Get language
    metadata_file = manuscript_folder / "metadata.json"
    if metadata_file.exists():
        with open(metadata_file) as f:
            language = json.load(f).get("language", "Unknown")
    else:
        language = "Unknown"
    
    for xml_file in manuscript_folder.glob("*.xml"):
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        for text_line in root.findall('.//{*}TextLine'):
            string_elem = text_line.find('.//{*}String')
            if string_elem is not None:
                text = string_elem.get('CONTENT', '')
                char_count = len(text)
                word_count = len(text.split())
                
                line_lengths.append(char_count)
                line_lengths_by_language[language].append(char_count)
                word_counts.append(word_count)

import statistics

print("\n=== Text Length Statistics (Characters) ===")
print(f"Total lines analyzed: {len(line_lengths)}")
print(f"Mean length: {statistics.mean(line_lengths):.1f}")
print(f"Median length: {statistics.median(line_lengths):.1f}")
print(f"Std dev: {statistics.stdev(line_lengths):.1f}")
print(f"Min length: {min(line_lengths)}")
print(f"Max length: {max(line_lengths)}")

# Percentiles
sorted_lengths = sorted(line_lengths)
print(f"\nPercentiles:")
print(f"  10th: {sorted_lengths[int(len(sorted_lengths)*0.1)]}")
print(f"  25th: {sorted_lengths[int(len(sorted_lengths)*0.25)]}")
print(f"  50th: {sorted_lengths[int(len(sorted_lengths)*0.5)]}")
print(f"  75th: {sorted_lengths[int(len(sorted_lengths)*0.75)]}")
print(f"  90th: {sorted_lengths[int(len(sorted_lengths)*0.9)]}")
print(f"  95th: {sorted_lengths[int(len(sorted_lengths)*0.95)]}")
print(f"  99th: {sorted_lengths[int(len(sorted_lengths)*0.99)]}")

print("\n=== Word Count Statistics ===")
print(f"Mean words per line: {statistics.mean(word_counts):.1f}")
print(f"Median words per line: {statistics.median(word_counts):.1f}")
print(f"Max words in a line: {max(word_counts)}")

print("\n=== Length Distribution by Language ===")
for lang in sorted(line_lengths_by_language.keys()):
    lengths = line_lengths_by_language[lang]
    print(f"{lang:<15} Mean: {statistics.mean(lengths):5.1f}  "
          f"Median: {statistics.median(lengths):5.1f}  "
          f"Max: {max(lengths):4d}")

# Check for problematic lines
print("\n=== Potential Issues ===")
empty_lines = sum(1 for l in line_lengths if l == 0)
very_short = sum(1 for l in line_lengths if l < 5)
very_long = sum(1 for l in line_lengths if l > 200)

print(f"Empty lines (0 chars): {empty_lines}")
print(f"Very short lines (<5 chars): {very_short}")
print(f"Very long lines (>200 chars): {very_long}")

if very_long > 0:
    print(f"\n⚠️  You have {very_long} very long lines (>200 chars)")
    print(f"   These might cause memory issues. Consider max_length parameter.")


=== Text Length Statistics (Characters) ===
Total lines analyzed: 151225
Mean length: 36.0
Median length: 36.0
Std dev: 17.5
Min length: 0
Max length: 292

Percentiles:
  10th: 14
  25th: 28
  50th: 36
  75th: 42
  90th: 55
  95th: 67
  99th: 93

=== Word Count Statistics ===
Mean words per line: 6.7
Median words per line: 7.0
Max words in a line: 53

=== Length Distribution by Language ===
Castilian       Mean:  36.9  Median:  36.0  Max:  292
Catalan         Mean:  44.0  Median:  41.0  Max:   76
French          Mean:  34.0  Median:  35.0  Max:   87
Gallician       Mean:  26.4  Median:  25.0  Max:   85
Italian         Mean:  36.6  Median:  35.0  Max:  128
Latin           Mean:  37.7  Median:  38.0  Max:  172
Navarrese       Mean:  27.0  Median:  27.0  Max:   37
Venitian        Mean:  36.0  Median:  48.0  Max:   64

=== Potential Issues ===
Empty lines (0 chars): 2384
Very short lines (<5 chars): 5835
Very long lines (>200 chars): 28

⚠️  You have 28 very long lines (>200 chars)
   The

### Image Quality Analysis

In [13]:
from pathlib import Path
from PIL import Image
import statistics

image_sizes = []
line_sizes = []
line_aspect_ratios = []
image_formats = []

# Sample 100 random manuscripts
import random
manuscripts = [d for d in dataset_path.iterdir() if d.is_dir()]
sample_manuscripts = random.sample(manuscripts, min(100, len(manuscripts)))

print("Analyzing 100 random manuscripts...")

for manuscript_folder in sample_manuscripts:
    for image_file in list(manuscript_folder.glob("*.jpg")) + list(manuscript_folder.glob("*.png")):
        try:
            img = Image.open(image_file)
            image_sizes.append(img.size)
            image_formats.append(img.format)
            
            # Get corresponding XML
            xml_file = image_file.parent / f"{image_file.stem}.xml"
            if xml_file.exists():
                import xml.etree.ElementTree as ET
                tree = ET.parse(xml_file)
                root = tree.getroot()
                
                for text_line in root.findall('.//{*}TextLine'):
                    width = float(text_line.get('WIDTH', 0))
                    height = float(text_line.get('HEIGHT', 0))
                    if width > 0 and height > 0:
                        line_sizes.append((width, height))
                        line_aspect_ratios.append(width / height)
        except Exception as e:
            print(f"Error processing {image_file}: {e}")

print("\n=== Page Image Statistics ===")
widths = [w for w, h in image_sizes]
heights = [h for w, h in image_sizes]
print(f"Sample size: {len(image_sizes)} images")
print(f"Width  - Mean: {statistics.mean(widths):.0f}, Min: {min(widths)}, Max: {max(widths)}")
print(f"Height - Mean: {statistics.mean(heights):.0f}, Min: {min(heights)}, Max: {max(heights)}")

from collections import Counter
print(f"\nMost common image sizes:")
for size, count in Counter(image_sizes).most_common(5):
    print(f"  {size[0]}×{size[1]}: {count} images")

print(f"\nImage formats: {Counter(image_formats)}")

print("\n=== Line Image Statistics ===")
line_widths = [w for w, h in line_sizes]
line_heights = [h for w, h in line_sizes]
print(f"Sample size: {len(line_sizes)} lines")
print(f"Line Width  - Mean: {statistics.mean(line_widths):.0f}, "
      f"Min: {min(line_widths):.0f}, Max: {max(line_widths):.0f}")
print(f"Line Height - Mean: {statistics.mean(line_heights):.0f}, "
      f"Min: {min(line_heights):.0f}, Max: {max(line_heights):.0f}")

print(f"\nLine Aspect Ratio (width/height):")
print(f"  Mean: {statistics.mean(line_aspect_ratios):.1f}")
print(f"  Median: {statistics.median(line_aspect_ratios):.1f}")
print(f"  Min: {min(line_aspect_ratios):.1f}")
print(f"  Max: {max(line_aspect_ratios):.1f}")

# Check for anomalies
very_tall = sum(1 for _, h in line_sizes if h > 150)
very_short = sum(1 for _, h in line_sizes if h < 30)
very_narrow = sum(1 for w, _ in line_sizes if w < 100)

print(f"\n=== Potential Issues ===")
print(f"Very tall lines (>150px): {very_tall}")
print(f"Very short lines (<30px): {very_short}")
print(f"Very narrow lines (<100px): {very_narrow}")

Analyzing 100 random manuscripts...

=== Page Image Statistics ===
Sample size: 646 images
Width  - Mean: 3379, Min: 1044, Max: 8676
Height - Mean: 4424, Min: 1812, Max: 8364

Most common image sizes:
  5202×7382: 33 images
  5191×7452: 27 images
  2509×3296: 26 images
  2412×3274: 26 images
  2448×3264: 20 images

Image formats: Counter({'JPEG': 592, 'PNG': 54})

=== Line Image Statistics ===
Sample size: 41233 lines
Line Width  - Mean: 1163, Min: 2, Max: 4975
Line Height - Mean: 124, Min: 3, Max: 776

Line Aspect Ratio (width/height):
  Mean: 9.8
  Median: 9.9
  Min: 0.1
  Max: 90.6

=== Potential Issues ===
Very tall lines (>150px): 11062
Very short lines (<30px): 21
Very narrow lines (<100px): 355


### Check for Duplicate Lines

In [14]:
# check_duplicates.py
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

all_texts = []
all_line_ids = []

for manuscript_folder in dataset_path.iterdir():
    if not manuscript_folder.is_dir():
        continue
    
    for xml_file in manuscript_folder.glob("*.xml"):
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        for text_line in root.findall('.//{*}TextLine'):
            line_id = text_line.get('ID')
            string_elem = text_line.find('.//{*}String')
            if string_elem is not None:
                text = string_elem.get('CONTENT', '')
                all_texts.append(text)
                all_line_ids.append(line_id)

print(f"\n=== Duplicate Analysis ===")
print(f"Total lines: {len(all_texts)}")
print(f"Unique texts: {len(set(all_texts))}")

text_counts = Counter(all_texts)
duplicates = {text: count for text, count in text_counts.items() if count > 1}

print(f"Duplicate text lines: {len(duplicates)}")
print(f"Total duplicate instances: {sum(duplicates.values()) - len(duplicates)}")

if duplicates:
    print(f"\nTop 10 most duplicated lines:")
    for text, count in sorted(duplicates.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  [{count}x] {text[:60]}...")

# Check line ID uniqueness
id_counts = Counter(all_line_ids)
duplicate_ids = {lid: count for lid, count in id_counts.items() if count > 1}

print(f"\n=== Line ID Uniqueness ===")
print(f"Total line IDs: {len(all_line_ids)}")
print(f"Unique line IDs: {len(set(all_line_ids))}")

if duplicate_ids:
    print(f"⚠️  WARNING: {len(duplicate_ids)} duplicate line IDs found!")
    print(f"This could cause issues in submission format!")
    for line_id, count in sorted(duplicate_ids.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {line_id}: appears {count} times")
else:
    print(f"✅ All line IDs are unique!")


=== Duplicate Analysis ===
Total lines: 151225
Unique texts: 144322
Duplicate text lines: 1696
Total duplicate instances: 6903

Top 10 most duplicated lines:
  [2384x] ...
  [167x] ....
  [145x] L...
  [131x] Q...
  [124x] A...
  [82x] E...
  [75x] D...
  [74x] I...
  [61x] C...
  [54x] S...

=== Line ID Uniqueness ===
Total line IDs: 151225
Unique line IDs: 126211
⚠️  WARNING: 634 duplicate line IDs found!
This could cause issues in submission format!
  line_8: appears 473 times
  line_6: appears 467 times
  line_12: appears 466 times
  line_11: appears 464 times
  line_7: appears 463 times


### Train/Val Split Verification

In [15]:
# verify_split_strategy.py
from pathlib import Path
import json
from collections import defaultdict

lines_per_manuscript = defaultdict(lambda: defaultdict(int))

for manuscript_folder in dataset_path.iterdir():
    if not manuscript_folder.is_dir():
        continue
    
    # Get language
    metadata_file = manuscript_folder / "metadata.json"
    if metadata_file.exists():
        with open(metadata_file) as f:
            language = json.load(f).get("language", "Unknown")
    else:
        continue
    
    # Count lines in this manuscript
    line_count = 0
    import xml.etree.ElementTree as ET
    for xml_file in manuscript_folder.glob("*.xml"):
        tree = ET.parse(xml_file)
        root = tree.getroot()
        line_count += len(root.findall('.//{*}TextLine'))
    
    lines_per_manuscript[language][manuscript_folder.name] = line_count

print("\n=== Manuscript Distribution by Language ===")
for lang in sorted(lines_per_manuscript.keys()):
    mss_dict = lines_per_manuscript[lang]
    total_mss = len(mss_dict)
    total_lines = sum(mss_dict.values())
    
    print(f"\n{lang}:")
    print(f"  Manuscripts: {total_mss}")
    print(f"  Total lines: {total_lines}")
    print(f"  Lines per manuscript: {total_lines/total_mss:.1f}")
    
    # Simulate 80/20 split at manuscript level
    train_mss_count = int(total_mss * 0.8)
    val_mss_count = total_mss - train_mss_count
    
    # Sort manuscripts by line count
    sorted_mss = sorted(mss_dict.items(), key=lambda x: x[1], reverse=True)
    
    # Assign to train (first 80%)
    train_lines = sum(count for _, count in sorted_mss[:train_mss_count])
    val_lines = sum(count for _, count in sorted_mss[train_mss_count:])
    
    print(f"  If split at manuscript level (80/20):")
    print(f"    Train: {train_mss_count} mss, {train_lines} lines ({train_lines/total_lines*100:.1f}%)")
    print(f"    Val:   {val_mss_count} mss, {val_lines} lines ({val_lines/total_lines*100:.1f}%)")
    
    if val_mss_count == 0:
        print(f"  ⚠️  WARNING: Not enough manuscripts for val split!")

print("\n=== Recommendation ===")
print("Split strategy: Manuscript-level (not line-level)")
print("Reasoning: Keeps lines from same manuscript together")
print("           Prevents data leakage between train/val")


=== Manuscript Distribution by Language ===

Castilian:
  Manuscripts: 52
  Total lines: 33805
  Lines per manuscript: 650.1
  If split at manuscript level (80/20):
    Train: 41 mss, 33365 lines (98.7%)
    Val:   11 mss, 440 lines (1.3%)

Catalan:
  Manuscripts: 6
  Total lines: 1597
  Lines per manuscript: 266.2
  If split at manuscript level (80/20):
    Train: 4 mss, 1215 lines (76.1%)
    Val:   2 mss, 382 lines (23.9%)

French:
  Manuscripts: 88
  Total lines: 58002
  Lines per manuscript: 659.1
  If split at manuscript level (80/20):
    Train: 70 mss, 55078 lines (95.0%)
    Val:   18 mss, 2924 lines (5.0%)

Gallician:
  Manuscripts: 1
  Total lines: 547
  Lines per manuscript: 547.0
  If split at manuscript level (80/20):
    Train: 0 mss, 0 lines (0.0%)
    Val:   1 mss, 547 lines (100.0%)

Italian:
  Manuscripts: 25
  Total lines: 8112
  Lines per manuscript: 324.5
  If split at manuscript level (80/20):
    Train: 20 mss, 7308 lines (90.1%)
    Val:   5 mss, 804 lines (9.

### Checking very long/short/narrow/wide lines

In [16]:
from pathlib import Path
import xml.etree.ElementTree as ET
from PIL import Image
import random

# Collect problematic lines
very_tall = []      # height > 150
very_short = []     # height < 30
very_narrow = []    # width < 100
very_wide = []      # aspect ratio > 20

for manuscript_folder in dataset_path.iterdir():
    if not manuscript_folder.is_dir():
        continue
    
    for xml_file in manuscript_folder.glob("*.xml"):
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        image_file = manuscript_folder / f"{xml_file.stem}.jpg"
        if not image_file.exists():
            image_file = manuscript_folder / f"{xml_file.stem}.png"
        
        if not image_file.exists():
            continue
        
        for text_line in root.findall('.//{*}TextLine'):
            width = float(text_line.get('WIDTH', 0))
            height = float(text_line.get('HEIGHT', 0))
            
            if height == 0:
                continue
            
            aspect_ratio = width / height
            line_id = text_line.get('ID')
            string_elem = text_line.find('.//{*}String')
            text = string_elem.get('CONTENT', '') if string_elem is not None else ''
            
            info = {
                'xml': xml_file,
                'image': image_file,
                'line_id': line_id,
                'width': width,
                'height': height,
                'aspect_ratio': aspect_ratio,
                'text': text,
                'hpos': float(text_line.get('HPOS', 0)),
                'vpos': float(text_line.get('VPOS', 0))
            }
            
            if height > 150:
                very_tall.append(info)
            elif height < 30:
                very_short.append(info)
            
            if width < 100:
                very_narrow.append(info)
            
            if aspect_ratio > 20:
                very_wide.append(info)

print(f"Found {len(very_tall)} very tall lines")
print(f"Found {len(very_short)} very short lines")
print(f"Found {len(very_narrow)} very narrow lines")
print(f"Found {len(very_wide)} very wide lines")

# Sample and save examples
def save_samples(lines, category_name, num_samples=10):
    samples = random.sample(lines, min(num_samples, len(lines)))
    output_dir = Path(f"problematic_lines/{category_name}")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    for i, line_info in enumerate(samples):
        try:
            page_img = Image.open(line_info['image'])
            h, v, w, ht = line_info['hpos'], line_info['vpos'], line_info['width'], line_info['height']
            
            # Crop line
            line_img = page_img.crop((h, v, h+w, v+ht))
            
            # Save with metadata
            filename = f"{category_name}_{i:02d}_h{int(ht)}_w{int(w)}_ar{line_info['aspect_ratio']:.1f}.jpg"
            line_img.save(output_dir / filename)
            
            # Save text
            with open(output_dir / f"{category_name}_{i:02d}.txt", 'w') as f:
                f.write(f"Text: {line_info['text']}\n")
                f.write(f"Height: {ht}px, Width: {w}px\n")
                f.write(f"Aspect ratio: {line_info['aspect_ratio']:.1f}\n")
                f.write(f"Source: {line_info['xml']}\n")
            
            print(f"Saved {filename}")
        except Exception as e:
            print(f"Error processing sample {i}: {e}")

# Save samples of each category
save_samples(very_tall, "very_tall", 20)
save_samples(very_short, "very_short", 10)
save_samples(very_narrow, "very_narrow", 10)
save_samples(very_wide, "very_wide", 10)

print("\n✅ Samples saved to problematic_lines/ directory")

Found 27670 very tall lines
Found 297 very short lines
Found 1815 very narrow lines
Found 1012 very wide lines
Saved very_tall_00_h152_w1231_ar8.1.jpg
Saved very_tall_01_h168_w1865_ar11.1.jpg
Saved very_tall_02_h203_w1853_ar9.1.jpg
Saved very_tall_03_h315_w1335_ar4.2.jpg
Saved very_tall_04_h199_w1859_ar9.3.jpg
Saved very_tall_05_h197_w1860_ar9.4.jpg
Saved very_tall_06_h151_w1245_ar8.2.jpg
Saved very_tall_07_h167_w1098_ar6.6.jpg
Saved very_tall_08_h215_w1853_ar8.6.jpg
Saved very_tall_09_h178_w1619_ar9.1.jpg
Saved very_tall_10_h154_w1603_ar10.4.jpg
Saved very_tall_11_h170_w1480_ar8.7.jpg
Saved very_tall_12_h162_w1202_ar7.4.jpg
Saved very_tall_13_h182_w1250_ar6.9.jpg
Saved very_tall_14_h305_w485_ar1.6.jpg
Saved very_tall_15_h180_w1846_ar10.3.jpg
Saved very_tall_16_h182_w1274_ar7.0.jpg
Saved very_tall_17_h284_w1840_ar6.5.jpg
Saved very_tall_18_h173_w1353_ar7.8.jpg
Saved very_tall_19_h199_w1901_ar9.6.jpg
Saved very_short_00_h29_w56_ar1.9.jpg
Saved very_short_01_h23_w108_ar4.7.jpg
Saved very